# Fine-tuning Qwen2.5-VL-7B-Instruct (QLoRA + early stopping)

El modelo más grande de la comparativa. Usamos QLoRA porque 7B en bf16 + LoRA + activaciones no cabe holgado en 12 GB.

Diferencias respecto al notebook del 3B:
- **`load_in_4bit = True`** (QLoRA)
- **`r = 64, alpha = 64`**: rank alto **necesario** porque la cuantización 4bit pierde precisión y LoRA tiene que compensar.
- `per_device_train_batch_size = 2`, `gradient_accumulation_steps = 4` → batch efectivo 8.
- Mismo early stopping (patience=1).

Resultados se guardan en `../results/qwen25_vl_7b.json`.

## 1. Setup

In [ ]:
import os
import json
import time
from pathlib import Path
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv()

import comet_ml

assert os.environ.get("HF_TOKEN", "").startswith("hf_"), "Falta HF_TOKEN en .env"
assert os.environ.get("COMET_API_KEY"), "Falta COMET_API_KEY en .env"
print("Tokens cargados.")

## 2. Descargar dataset

In [ ]:
from huggingface_hub import snapshot_download

HF_DATASET = "damianGil/wildfire-prevention"

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data" / "wildfire"
RESULTS_DIR  = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=HF_DATASET,
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print(f"Dataset listo en {DATA_DIR}")

## 3. Cargar modelo Qwen2.5-VL-7B (QLoRA, en 4 bits)

In [ ]:
from unsloth import FastVisionModel
import torch

MODEL_NAME  = "unsloth/Qwen2.5-VL-7B-Instruct"
MODEL_SHORT = "qwen25_vl_7b"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 4096,
    load_in_4bit   = True,    # QLoRA
)

print(f"Modelo cargado: {MODEL_NAME}")
print(f"VRAM reservada: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

## 4. Configurar LoRA (r=64, alpha=64 — rank alto requerido por QLoRA)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r            = 64,
    lora_alpha   = 64,
    lora_dropout = 0.05,
    bias         = "none",
    random_state = 42,
    use_rslora   = False,
    loftq_config = None,
)
model.print_trainable_parameters()

## 5. Preparar dataset

In [ ]:
from datasets import load_dataset
from PIL import Image

ds = load_dataset(str(DATA_DIR))
print(f"Train: {len(ds['train'])}  |  Test: {len(ds['test'])}")

SYSTEM_PROMPT = """\
You are a remote sensing analyst specialising in wildfire risk assessment.
You will be given two Sentinel-2 satellite images of the same land tile:
  1. RGB composite (bands B4-B3-B2): natural colour, useful for terrain, infrastructure, and land cover.
  2. SWIR composite (bands B12-B8-B4): highlights vegetation moisture stress and dryness. Healthy vegetation appears green/cyan, stressed or dry vegetation appears orange/red, bare soil appears magenta/pink, and burned areas appear dark red or black.

Assess the wildfire risk of the tile and return ONLY a valid JSON object — no markdown, no explanation outside the JSON — with exactly these fields:

{
  "risk_level": "low | medium | high",
  "dry_vegetation_present": true | false,
  "urban_interface": true | false,
  "steep_terrain": true | false,
  "water_body_present": true | false,
  "image_quality_limited": true | false
}
"""

USER_TEXT = (
    "Image 1 is the RGB composite. Image 2 is the SWIR composite. "
    "Return the wildfire risk JSON for this tile."
)
PROMPT = f"{SYSTEM_PROMPT.strip()}\n\n{USER_TEXT}"


def to_unsloth_conversation(sample):
    rgb  = Image.open(DATA_DIR / sample["rgb_path"]).convert("RGB")
    swir = Image.open(DATA_DIR / sample["swir_path"]).convert("RGB")
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": rgb},
                {"type": "image", "image": swir},
                {"type": "text",  "text":  PROMPT},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["output"]},
            ]},
        ]
    }


train_dataset = [to_unsloth_conversation(s) for s in ds["train"]]
test_dataset  = [to_unsloth_conversation(s) for s in ds["test"]]
print(f"Conversaciones preparadas. Train={len(train_dataset)}, Test={len(test_dataset)}")

## 6. Trainer + entrenamiento (con early stopping)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=1,
                                           early_stopping_threshold = 0.001,)],
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 2,
        warmup_ratio                = 0.03,
        learning_rate               = 1e-4,
        lr_scheduler_type           = "cosine",
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        seed                        = 42,
        output_dir                  = f"outputs/{MODEL_SHORT}",
        run_name                    = f"{MODEL_SHORT}-qlora-r64-vision-on-early",
        report_to                   = ["comet_ml"],

        eval_strategy               = "epoch",
        per_device_eval_batch_size  = 2,
        save_strategy               = "epoch",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,

        remove_unused_columns = False,
        dataset_text_field    = "",
        dataset_kwargs        = {"skip_prepare_dataset": True},
        max_length            = 4096,
    ),
)

In [ ]:
torch.cuda.reset_peak_memory_stats()
start_mem = torch.cuda.memory_reserved() / 1024**3
print(f"VRAM antes de train: {start_mem:.2f} GB")

trainer_stats = trainer.train()

peak_mem = torch.cuda.max_memory_reserved() / 1024**3
actual_epochs = trainer_stats.metrics.get('epoch', 0)
print(f"\nTraining: {trainer_stats.metrics['train_runtime']/60:.2f} min")
print(f"Epochs reales: {actual_epochs:.1f}")
print(f"VRAM pico: {peak_mem:.2f} GB")

## 7. Evaluación completa

In [ ]:
FastVisionModel.for_inference(model)

FIELDS = ["risk_level", "dry_vegetation_present", "urban_interface",
          "steep_terrain", "water_body_present", "image_quality_limited"]

N_EVAL = len(ds["test"])
correct = defaultdict(int)
valid_json_count = 0
inf_times = []

for i in range(N_EVAL):
    s = ds["test"][i]
    rgb  = Image.open(DATA_DIR / s["rgb_path"]).convert("RGB")
    swir = Image.open(DATA_DIR / s["swir_path"]).convert("RGB")
    gt   = json.loads(s["output"])

    msgs = [{"role": "user", "content": [
        {"type": "image"}, {"type": "image"}, {"type": "text", "text": PROMPT}
    ]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inp  = tokenizer([rgb, swir], text, add_special_tokens=False, return_tensors="pt").to("cuda")

    t0 = time.time()
    out_ids = model.generate(**inp, max_new_tokens=256, use_cache=True,
                              temperature=0.1, min_p=0.1)
    inf_times.append(time.time() - t0)

    pred_text = tokenizer.decode(out_ids[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        clean = pred_text.strip()
        if clean.startswith("```"):
            clean = clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        pred = json.loads(clean)
        valid_json_count += 1
        for f in FIELDS:
            if pred.get(f) == gt.get(f):
                correct[f] += 1
    except (json.JSONDecodeError, IndexError):
        pass

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{N_EVAL} muestras evaluadas")

valid_json    = valid_json_count / N_EVAL
accuracy_per_field = {f: correct[f] / N_EVAL for f in FIELDS}
overall = sum(correct.values()) / (N_EVAL * len(FIELDS))
avg_inf_time = sum(inf_times) / len(inf_times)

print(f"\n=== RESULTADOS Qwen2.5-VL-7B (QLoRA, r=64, early stop) ===")
print(f"valid_json:  {valid_json:.3f}")
for f in FIELDS:
    print(f"  {f:25s} {accuracy_per_field[f]:.3f}")
print(f"\noverall: {overall:.3f}")
print(f"tiempo medio inferencia/muestra: {avg_inf_time:.2f} s")

## 8. Guardar resultados

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

results = {
    "model_name":  MODEL_NAME,
    "model_short": MODEL_SHORT,
    "training": {
        "num_params_total":      total_params,
        "num_params_trainable":  trainable_params,
        "vram_peak_gb":          peak_mem,
        "training_time_min":     trainer_stats.metrics['train_runtime'] / 60,
        "epochs_actual":         trainer_stats.metrics.get('epoch', None),
        "loss_train_final":      trainer_stats.training_loss,
        "config": {
            "r": 64, "lora_alpha": 64, "lora_dropout": 0.05,
            "finetune_vision_layers": True,
            "load_in_4bit":           True,
            "num_train_epochs_max":   5,
            "early_stopping_patience": 1,
            "batch_effective":        8,
            "learning_rate":          1e-4,
            "lr_scheduler":           "cosine",
            "warmup_ratio":           0.03,
        },
    },
    "eval": {
        "num_samples":           N_EVAL,
        "valid_json":            valid_json,
        "accuracy_per_field":    accuracy_per_field,
        "overall":               overall,
        "inference_time_per_sample_s": avg_inf_time,
    },
}

out_path = RESULTS_DIR / f"{MODEL_SHORT}.json"
out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Resultados guardados en: {out_path}")
print(json.dumps(results, indent=2))